# STARD Dataset → BEIR Format

Convert [STARD](https://github.com/oneal2000/STARD) (Chinese Statute Retrieval Dataset) to BEIR format.

- **Task**: Statute retrieval from non-professional legal queries
- **Language**: Chinese
- **Splits**: Use provided train/dev split (dev as test)

In [1]:
import subprocess, os

STARD_DIR = os.path.join("..", "data", "_raw", "STARD")

if not os.path.exists(STARD_DIR):
    os.makedirs(os.path.join("..", "data", "_raw"), exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/oneal2000/STARD.git", STARD_DIR],
        check=True
    )
    print("Cloned STARD repo")
else:
    print(f"STARD already exists at {STARD_DIR}")

print(os.listdir(os.path.join(STARD_DIR, "data")))

Cloned STARD repo
['corpus.jsonl', 'example', 'queries.json']


In [2]:
import json

# load queries.json
with open(os.path.join(STARD_DIR, "data", "queries.json"), encoding="utf-8") as f:
    queries_raw = json.load(f)

print(f"Total queries: {len(queries_raw)}")
print(f"\nSample query:")
sample = queries_raw[0]
for k, v in sample.items():
    val = repr(v)[:200]
    print(f"  {k}: {val}")

Total queries: 1543

Sample query:
  query_id: 0
  问题: '谁可以成为个体工商户？'
  相关法规: {'中华人民共和国民法典第五十四条': '自然人从事工商业经营，经依法登记，为个体工商户。个体工商户可以起字号。\n', '个体工商户条例第二条': '有经营能力的公民，依照本条例规定经工商行政管理部门登记，从事工商业经营的，为个体工商户。\n\n个体工商户可以个人经营，也可以家庭经营。\n\n个体工商户的合法权益受法律保护，任何单位和个人不得侵害。\n'}
  match_id: [705, 49282]
  match_name: ['中华人民共和国民法典第五十四条', '个体工商户条例第二条']


In [3]:
# load corpus.jsonl
corpus_raw = []
with open(os.path.join(STARD_DIR, "data", "corpus.jsonl"), encoding="utf-8") as f:
    for line in f:
        corpus_raw.append(json.loads(line))

print(f"Total corpus docs: {len(corpus_raw)}")
print(f"\nSample corpus entry:")
for k, v in corpus_raw[0].items():
    val = repr(v)[:200]
    print(f"  {k}: {val}")

Total corpus docs: 55348

Sample corpus entry:
  id: 0
  name: '中华人民共和国民法典第四百六十三条'
  content: '本编调整因合同产生的民事关系。\\n'


In [4]:
# load train/dev split from example files
def load_query_ids(filepath):
    ids = set()
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if parts:
                ids.add(int(parts[0]))
    return ids

example_dir = os.path.join(STARD_DIR, "data", "example")
train_ids = load_query_ids(os.path.join(example_dir, "train.query.txt"))
dev_ids = load_query_ids(os.path.join(example_dir, "dev.query.txt"))

print(f"Train query IDs: {len(train_ids)}")
print(f"Dev query IDs (→ test): {len(dev_ids)}")
print(f"Overlap: {len(train_ids & dev_ids)}")
print(f"Coverage: {len(train_ids | dev_ids)} / {len(queries_raw)}")

Train query IDs: 1235
Dev query IDs (→ test): 308
Overlap: 0
Coverage: 1543 / 1543


## Convert to BEIR format

Target files under `../data/stard/`:
- `corpus.jsonl` — 55,348 statute articles
- `queries.jsonl` — all queries
- `qrels_train.tsv` / `qrels_test.tsv` — dev used as test
- `dataset_stats.json`

In [5]:
OUTPUT_DIR = os.path.join("..", "data", "stard")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- corpus.jsonl ---
corpus_path = os.path.join(OUTPUT_DIR, "corpus.jsonl")
with open(corpus_path, "w", encoding="utf-8") as f:
    for doc in corpus_raw:
        entry = {
            "_id": str(doc["id"]),
            "title": doc["name"],
            "text": doc["content"],
        }
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Wrote {len(corpus_raw)} documents to {corpus_path}")

Wrote 55348 documents to ..\data\stard\corpus.jsonl


In [6]:
# --- queries.jsonl + qrels ---
# build corpus id set for validation
corpus_ids = {str(doc["id"]) for doc in corpus_raw}

queries_path = os.path.join(OUTPUT_DIR, "queries.jsonl")
qrels_train_path = os.path.join(OUTPUT_DIR, "qrels_train.tsv")
qrels_test_path = os.path.join(OUTPUT_DIR, "qrels_test.tsv")

n_train_judgments = 0
n_test_judgments = 0
n_missing_docs = 0
n_unassigned = 0

# question field is in Chinese: "问题"
QUESTION_KEY = "\u95ee\u9898"  # 问题

with open(queries_path, "w", encoding="utf-8") as fq, \
     open(qrels_train_path, "w", encoding="utf-8") as ft, \
     open(qrels_test_path, "w", encoding="utf-8") as fte:
    
    ft.write("query_id\tdoc_id\tscore\n")
    fte.write("query_id\tdoc_id\tscore\n")
    
    for q in queries_raw:
        qid_num = q["query_id"]
        qid = f"q{qid_num}"
        
        query_entry = {
            "_id": qid,
            "text": q[QUESTION_KEY],
            "metadata": {},
        }
        fq.write(json.dumps(query_entry, ensure_ascii=False) + "\n")
        
        # determine split
        if qid_num in train_ids:
            qrels_file = ft
            is_train = True
        elif qid_num in dev_ids:
            qrels_file = fte
            is_train = False
        else:
            # query not in either split — assign to train
            qrels_file = ft
            is_train = True
            n_unassigned += 1
        
        for doc_id in q.get("match_id", []):
            doc_id_str = str(doc_id)
            if doc_id_str not in corpus_ids:
                n_missing_docs += 1
                continue
            qrels_file.write(f"{qid}\t{doc_id_str}\t1\n")
            if is_train:
                n_train_judgments += 1
            else:
                n_test_judgments += 1

print(f"Wrote {len(queries_raw)} queries to {queries_path}")
print(f"Train judgments: {n_train_judgments}")
print(f"Test judgments: {n_test_judgments}")
print(f"Missing doc references: {n_missing_docs}")
print(f"Unassigned queries (added to train): {n_unassigned}")

Wrote 1543 queries to ..\data\stard\queries.jsonl
Train judgments: 2205
Test judgments: 512
Missing doc references: 0
Unassigned queries (added to train): 0


In [7]:
# --- dataset_stats.json ---
total_judgments = n_train_judgments + n_test_judgments
n_queries = len(queries_raw)
n_train_queries = len(train_ids) + n_unassigned
n_test_queries = len(dev_ids)

stats = {
    "dataset": "stard",
    "language": "zh",
    "num_documents": len(corpus_raw),
    "num_queries": n_queries,
    "num_train_queries": n_train_queries,
    "num_test_queries": n_test_queries,
    "num_relevance_judgments": total_judgments,
    "num_train_judgments": n_train_judgments,
    "num_test_judgments": n_test_judgments,
    "avg_relevant_docs_per_query": total_judgments / n_queries if n_queries > 0 else 0,
    "note": "dev split used as test; queries not in train/dev assigned to train",
}

stats_path = os.path.join(OUTPUT_DIR, "dataset_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(json.dumps(stats, indent=2))

{
  "dataset": "stard",
  "language": "zh",
  "num_documents": 55348,
  "num_queries": 1543,
  "num_train_queries": 1235,
  "num_test_queries": 308,
  "num_relevance_judgments": 2717,
  "num_train_judgments": 2205,
  "num_test_judgments": 512,
  "avg_relevant_docs_per_query": 1.7608554763447828,
  "note": "dev split used as test; queries not in train/dev assigned to train"
}


In [8]:
# --- sanity check ---
print("=== Verification ===")

with open(os.path.join(OUTPUT_DIR, "corpus.jsonl"), encoding="utf-8") as f:
    first_doc = json.loads(f.readline())
    print(f"corpus.jsonl first entry: _id={first_doc['_id']}, title={first_doc['title'][:60]}")

with open(os.path.join(OUTPUT_DIR, "queries.jsonl"), encoding="utf-8") as f:
    first_query = json.loads(f.readline())
    print(f"queries.jsonl first entry: _id={first_query['_id']}, text={first_query['text'][:80]}")

for split in ["train", "test"]:
    path = os.path.join(OUTPUT_DIR, f"qrels_{split}.tsv")
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    print(f"qrels_{split}.tsv: {len(lines)-1} judgments (excl. header)")

print(f"\nAll files written to {os.path.abspath(OUTPUT_DIR)}")

=== Verification ===
corpus.jsonl first entry: _id=0, title=中华人民共和国民法典第四百六十三条
queries.jsonl first entry: _id=q0, text=谁可以成为个体工商户？
qrels_train.tsv: 2205 judgments (excl. header)
qrels_test.tsv: 512 judgments (excl. header)

All files written to c:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\stard
